# Single-EV Droplet Occupancy Calculator

**Interactive notebook** for the companion calculator of:
> *Droplet microfluidics for single-extracellular-vesicle analysis*

Repository: [github.com/cenab-ujep/single-ev-droplet-calculator](https://github.com/cenab-ujep/single-ev-droplet-calculator)

No installation needed — just run each cell with **Shift + Enter**.

## Setup
Install the calculator directly from GitHub (takes a few seconds).

In [ ]:
!pip install -q git+https://github.com/cenab-ujep/single-ev-droplet-calculator.git@v0.1.0

## 1. Occupancy metrics at a single operating point

Change `LAMBDA` below to explore different loading parameters.

In [ ]:
from single_ev_droplet_calculator import occupancy_metrics

LAMBDA = 0.10  # <-- change this value

m = occupancy_metrics(LAMBDA)
print(f"lambda              = {m.lambda_value:.6f}")
print(f"P_empty             = {m.p_empty:.6f}")
print(f"P_single            = {m.p_single:.6f}")
print(f"P_multi>=2          = {m.p_multi_ge_2:.6f}")
print(f"Purity (occupied)   = {m.purity_given_occupied:.6f}")

## 2. Compare two operating points (manuscript anchors)

In [ ]:
from single_ev_droplet_calculator import compare_operating_points

c = compare_operating_points(lambda_a=0.10, lambda_b=0.20)
print(f"Single-event yield ratio  (b/a) = {c.single_yield_ratio_b_over_a:.6f}")
print(f"Multi-occupancy burden ratio     = {c.multi_burden_ratio_b_over_a:.6f}")

## 3. Plan: how many droplets do I need?

Given a target number of interpreted single-EV events, compute the
required droplet count after QC and identity filtering.

In [ ]:
from single_ev_droplet_calculator import required_droplets_for_target_events

n = required_droplets_for_target_events(
    target_interpretable_single_events=10_000,
    lambda_value=0.10,
    q_qc=0.60,
    q_identity=0.70,
)
print(f"Required droplets: {n:,.0f}")

## 4. Estimate lambda from empty-droplet counts

If you observe 9,048 empty droplets out of 10,000 total, what is
the estimated lambda and its 95% confidence interval?

In [ ]:
from single_ev_droplet_calculator import (
    lambda_from_empty_fraction,
    lambda_confidence_interval_from_empty_counts,
)

lam_hat = lambda_from_empty_fraction(9048 / 10000)
ci = lambda_confidence_interval_from_empty_counts(n_empty=9048, n_total=10000)

print(f"Estimated lambda = {lam_hat:.6f}")
print(f"95% CI           = [{ci.lower:.6f}, {ci.upper:.6f}]")

## 5. Visualise the purity-throughput tradeoff

Reproduces the core relationship shown in the manuscript (Figure 1B).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambdas = np.linspace(0.001, 1.0, 500)
p_single = lambdas * np.exp(-lambdas)
p_empty = np.exp(-lambdas)
purity = p_single / (1.0 - p_empty)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(p_single, purity, color="#009E73", lw=2)

# Mark manuscript anchors
for lam, color in [(0.10, "#CC79A7"), (0.20, "#CC79A7")]:
    ps = lam * np.exp(-lam)
    pur = ps / (1.0 - np.exp(-lam))
    ax.scatter(ps, pur, color=color, s=50, zorder=3)
    ax.annotate(f"  $\\lambda$={lam:.2f}", xy=(ps, pur), fontsize=9)

ax.set_xlabel("Effective single-event throughput ($P_{single}$)")
ax.set_ylabel("Purity among occupied droplets")
ax.set_title("Purity vs Throughput")
ax.set_ylim(0.65, 1.01)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
*Calculator v0.1.0 — MIT License*